# 07 — Xuất ONNX một input / một output



In [ ]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

In [1]:
# Nạp định nghĩa từ 03 mà không chạy train, giống 04/05/06.
from pathlib import Path

_cwd = Path.cwd().resolve()
_ROOT = next((p for p in (_cwd, *_cwd.parents)
              if (p / "Code" / "03_train_model.ipynb").is_file()), None)
assert _ROOT is not None, f"Không thấy Code/03_train_model.ipynb quanh {_cwd}"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_ROOT / "Code" / "03_train_model.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import json
import warnings

import joblib
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from onnx import TensorProto as TP
from onnx import compose, helper
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

Gốc dự án: D:\12.VQEC\02.SDC
Đã nạp hàm SDC từ 03_train_model.ipynb


## 1. Hằng số hợp đồng

Tên input/output, nhãn abstain và opset. `ONNX_MODES` / `ONNX_KEY_SEP` phải khớp
`FingerprintTable.MODES` và `KEY_SEP` ở 03 — lệch một ký tự là tầng L1 trượt sạch mà
không báo lỗi, nên mục 8 đo parity chứ không tin vào đọc code.

In [2]:
ONNX_INPUT = "input"
ONNX_OUTPUT = "output"
ONNX_ABSTAIN = "unknown"          # nhãn trả về khi policy từ chối
ONNX_FILE = "sdc_multihead.onnx"
ONNX_CONTRACT_VERSION = "3.0.0"   # 2.x: 40 input + 2 output; 3.x: 1 input + 1 output

ONNX_OPSET = 20      # StringConcat là op của opset 20; dưới đó không dựng được khoá L1
ONNX_ML_OPSET = 3

# Locale cho `StringNormalizer`. skl2onnx không đặt thuộc tính này, mà onnxruntime khi đó
# lấy mặc định `en_US.UTF-8` và dựng `std::locale` NGAY LÚC khởi tạo session. Router
# OpenWrt chạy musl chỉ có `C`, nên model chết trước khi chạy được dòng nào:
#
#   Failed to construct locale with name:en_US.UTF-8:locale::facet::_S_create_c_locale
#
# `LANG=C` / `LC_ALL=C` không cứu được vì tên locale nằm trong thuộc tính của node, không
# phải lấy từ môi trường. Mục 7 ghi `C` cho mọi node, mục 9 đo parity.
ONNX_LOCALE = "C"

# Duyệt từ mode cụ thể nhất tới chung nhất, y như `FingerprintTable.MODES`.
ONNX_MODES = [(mode, cols) for mode, cols, _flags in FingerprintTable.MODES]
ONNX_KEY_SEP = FingerprintTable.KEY_SEP
ONNX_PAIR_SEP = "\x1f"            # nối (model, type) để tra cặp hợp lệ của hierarchy
ONNX_NEVER_KEY = "\x00__sdc_never__"   # khoá giữ chỗ cho LabelEncoder có bảng rỗng

# `Predictor._model_scores` chấm ngưỡng trên float64 của sklearn, còn graph cộng dồn
# float32 trong TreeEnsemble. Chênh lệch cỡ 1e-7 đủ lật mọi ca `threshold = 1.0` — và
# bảng ngưỡng của run hiện tại có ba ô đúng bằng 1.0. Dùng lại đúng dung sai mà
# `Predictor._has_policy_violation` đã công nhận, để answered khớp nhau.
ONNX_EPS = 5e-5

## 2. Bộ dựng graph

Gom node và initializer, sinh tên duy nhất. Không giữ trạng thái nào khác — mọi quyết
định về hình dạng graph nằm ở các hàm `build_*` bên dưới.

In [3]:
class OnnxGraphBuilder:
    """Gom node + initializer cho một graph, tự sinh tên không đụng nhau."""

    def __init__(self, prefix="sdc"):
        self.prefix = prefix
        self.nodes = []
        self.inits = []
        self._n = 0

    def name(self, stem):
        self._n += 1
        return f"{self.prefix}/{stem}_{self._n}"

    def add(self, op_type, inputs, out_stem, **attrs):
        out = self.name(out_stem)
        domain = attrs.pop("domain", "")
        self.nodes.append(helper.make_node(op_type, list(inputs), [out],
                                           name=self.name(f"n_{op_type}"),
                                           domain=domain, **attrs))
        return out

    def const(self, array, stem, dtype=None):
        """Initializer từ giá trị numpy. Chuỗi được encode utf-8."""
        out = self.name(stem)
        if dtype == TP.STRING:
            values = np.asarray(array, dtype=object)
            self.inits.append(helper.make_tensor(
                out, TP.STRING, list(values.shape),
                [s.encode("utf-8") for s in values.ravel()]))
        else:
            self.inits.append(onnx.numpy_helper.from_array(np.asarray(array), out))
        return out

    def str_const(self, value, stem="str"):
        return self.const(np.array([[value]], dtype=object), stem, dtype=TP.STRING)

    def label_encode(self, key, keys, stem, values, default):
        """LabelEncoder chịu được bảng rỗng.

        `ai.onnx.ml.LabelEncoder` bắt buộc có ít nhất một khoá, mà vài ô (mode, head) của
        sổ nhập nhằng đúng là rỗng. Nhồi một khoá sentinel không thể xuất hiện trong dữ
        liệu thật, để giữ đúng shape mà không đổi kết quả.
        """
        keys = [str(k) for k in keys] or [ONNX_NEVER_KEY]
        values = list(values) or [default]
        attrs = ({"default_string": default, "values_strings": [str(v) for v in values]}
                 if isinstance(default, str)
                 else {"default_int64": default, "values_int64s": [int(v) for v in values]})
        return self.add("LabelEncoder", [key], stem, domain="ai.onnx.ml",
                        keys_strings=keys, **attrs)

    def concat_str(self, parts, stem="key"):
        """StringConcat nhị phân, nối tuần tự. Phần tử bọc `onnx_lit` là chuỗi literal."""
        acc = None
        for part in parts:
            if isinstance(part, str) and part.startswith("\0lit"):
                part = self.str_const(part[4:])
            acc = part if acc is None else self.add("StringConcat", [acc, part], stem)
        return acc

    def any_of(self, flags, stem):
        acc = None
        for flag in flags:
            acc = flag if acc is None else self.add("Or", [acc, flag], stem)
        return acc


def onnx_lit(text):
    """Đánh dấu một chuỗi literal cho `concat_str`."""
    return "\0lit" + text

## 3. Front — một tensor chuỗi thành các cột rời

`input` là **một** tensor string `[N, 40]`, tách bằng `Split` rồi `Cast` các cột numeric
sang float32. Cast chuỗi→float là khác biệt duy nhất so với contract 2.x: user space gửi
`"1"` / `"0"` thay vì số. Đổi lại chỉ còn một tensor, nên **không thể lệch thứ tự input**
— lỗi mà giao diện 40 input rời rất dễ mắc và không có cách nào phát hiện lúc chạy.

In [4]:
def onnx_build_front(gb, cols, num_cols):
    """`input` string [N,40] -> dict {tên cột: tensor}. Cột numeric đã Cast sang float32."""
    outs = [gb.name(f"col/{c}") for c in cols]
    gb.nodes.append(helper.make_node("Split", [ONNX_INPUT], outs,
                                     name=gb.name("n_Split"), axis=1,
                                     num_outputs=len(cols)))
    num = set(num_cols)
    return {col: (gb.add("Cast", [t], f"num/{col}", to=TP.FLOAT) if col in num else t)
            for col, t in zip(cols, outs)}

## 4. Tầng L1 — tra bảng vân tay

Lặp lại `FingerprintTable.lookup`: duyệt mode từ cụ thể tới chung, và khoá **nhập nhằng**
ở một mode thì DỪNG chứ không rơi xuống mode chung hơn (mode chung hơn không thể tách
được cái mà mode cụ thể hơn đã không tách nổi).

Không cần kiểm `has_*` hay `<missing>` như bản python: `aggregate` đặt `<missing>` cho
nguồn vắng mặt, mà khoá trong bảng chỉ sinh từ hàng usable nên không khoá nào chứa
`<missing>`. Dựng khoá vô điều kiện vì thế luôn trượt đúng những chỗ bản python bỏ qua —
mục 8 xác nhận điều này chứ không chỉ lập luận.

In [5]:
def onnx_build_l1(gb, columns, tables, ambiguous, head):
    """Trả tensor nhãn L1 cho một head; chuỗi rỗng nghĩa là không có nhãn nào."""
    label = gb.str_const("")
    stopped = None
    for mode, key_cols in ONNX_MODES:
        parts = []
        for i, col in enumerate(key_cols):
            if i:
                parts.append(onnx_lit(ONNX_KEY_SEP))
            parts.append(columns[col])
        key = gb.concat_str(parts, f"l1key/{mode}")

        book = tables.get((mode, head), {})
        amb = ambiguous.get((mode, head), {})
        hit = gb.label_encode(key, book, f"l1hit/{mode}/{head}",
                              values=book.values(), default="")
        amb_flag = gb.add("Cast", [gb.label_encode(
            key, amb, f"l1amb/{mode}/{head}", values=[1] * len(amb), default=0)],
            "l1ambflag", to=TP.BOOL)
        has_hit = gb.add("Not", [gb.add("Equal", [hit, gb.str_const("")], "l1empty")],
                         "l1has")

        if stopped is None:
            label = gb.add("Where", [has_hit, hit, label], f"l1label/{mode}")
            stopped = gb.add("Or", [has_hit, amb_flag], "l1stop")
        else:
            # Mode trước đã chốt (hit hoặc nhập nhằng) -> không xét mode này nữa.
            fresh = gb.add("Not", [stopped], "l1open")
            take = gb.add("And", [fresh, has_hit], "l1take")
            label = gb.add("Where", [take, hit, label], f"l1label/{mode}")
            newly = gb.add("And", [fresh, gb.add("Or", [has_hit, amb_flag], "l1any")],
                           "l1new")
            stopped = gb.add("Or", [stopped, newly], "l1stop")
    return label

## 5. Quyết định từng head, rồi hierarchy

Đúng thứ tự của `Predictor.predict_row`:

1. **L1 có nhãn** → trả lời nếu chính L2 còn đỡ nhãn đó ở mức `l1_floor`. Vân tay khớp
   nhưng model không đồng ý thì đó là va chạm vân tay, không phải bằng chứng.
2. **L1 trượt** → cần đủ `min_sources` nguồn, và confidence vượt ngưỡng của **đúng số
   nguồn đó** (`thresholds_by_source`).
3. **hierarchy** → `model` mâu thuẫn với `make`/`type` thì hạ **cả cụm** xuống abstain.

`Where` của onnxruntime không nhận nhánh bool, nên chỗ chọn giữa hai cờ được viết thành
`(c AND a) OR (NOT c AND b)`.

In [6]:
def onnx_build_head(gb, head, probs, l2_label, l1_label, classes, n_sources,
                    thresholds, min_sources, l1_floor):
    """Quyết định của một head, trước hierarchy. Trả (tensor nhãn, tensor cờ đã trả lời)."""
    axes = gb.const(np.array([1], np.int64), "axes")
    maxp = gb.add("ReduceMax", [probs, axes], f"maxp/{head}", keepdims=1)

    # --- nhánh L2: đủ nguồn, và confidence vượt ngưỡng ứng với đúng số nguồn đó
    enough = gb.add("GreaterOrEqual",
                    [n_sources, gb.const(np.float32([[min_sources[head]]]), "minsrc")],
                    f"enough/{head}")
    table = np.float32([thresholds[f"{head}|{k}"] for k in (1, 2, 3, 4)])
    idx = gb.add("Cast", [n_sources], "nsrc_i", to=TP.INT64)
    idx = gb.add("Sub", [idx, gb.const(np.int64([[1]]), "one")], "nsrc_i0")
    idx = gb.add("Clip", [idx, gb.const(np.int64(0), "lo"), gb.const(np.int64(3), "hi")],
                 "nsrc_ic")
    thr = gb.add("Gather", [gb.const(table, f"thr/{head}"), idx], f"thr/{head}", axis=0)
    thr = gb.add("Reshape", [thr, gb.const(np.int64([-1, 1]), "shape")], "thr2d")
    over = gb.add("GreaterOrEqual",
                  [gb.add("Add", [maxp, gb.const(np.float32([[ONNX_EPS]]), "eps")],
                          "maxp_e"), thr],
                  f"over/{head}")
    ans_l2 = gb.add("And", [enough, over], f"ansl2/{head}")

    # --- nhánh L1: nhãn exact-match chỉ được trả lời nếu L2 còn đỡ nó tới `l1_floor`
    has_l1 = gb.add("Not", [gb.add("Equal", [l1_label, gb.str_const("")], "l1e")],
                    f"hasl1/{head}")
    cls_idx = gb.label_encode(l1_label, [str(c) for c in classes], f"l1idx/{head}",
                              values=range(len(classes)), default=-1)
    safe = gb.add("Max", [cls_idx, gb.const(np.int64([[0]]), "zero")], "l1idx_safe")
    backing = gb.add("GatherElements", [probs, safe], f"l1back/{head}", axis=1)
    in_model = gb.add("GreaterOrEqual", [cls_idx, gb.const(np.int64([[0]]), "zero")],
                      f"l1inmodel/{head}")
    floor_ok = gb.add("GreaterOrEqual",
                      [gb.add("Add", [backing, gb.const(np.float32([[ONNX_EPS]]), "eps")],
                              "back_e"),
                       gb.const(np.float32([[l1_floor]]), "floor")], f"l1floor/{head}")
    ans_l1 = gb.add("And", [in_model, floor_ok], f"ansl1/{head}")

    label = gb.add("Where", [has_l1, l1_label, l2_label], f"pred/{head}")
    no_l1 = gb.add("Not", [has_l1], f"nol1/{head}")
    answered = gb.add("Or", [gb.add("And", [has_l1, ans_l1], "pick_l1"),
                             gb.add("And", [no_l1, ans_l2], "pick_l2")], f"ans/{head}")
    return label, answered


def onnx_build_hierarchy(gb, hierarchy, labels, answered):
    """`Predictor._apply_hierarchy`: model mâu thuẫn với make/type -> hạ CẢ CỤM."""
    if not hierarchy:
        return answered

    model_label, model_ans = labels[MODEL_HEAD], answered[MODEL_HEAD]
    keys = sorted(hierarchy)
    has_rule = gb.add("Cast", [gb.label_encode(model_label, keys, "hier/rule",
                                               values=[1] * len(keys), default=0)],
                      "hier/hasrule", to=TP.BOOL)
    has_rule = gb.add("And", [has_rule, model_ans], "hier/active")

    rule_make = gb.label_encode(
        model_label, keys, "hier/make",
        values=[str(hierarchy[k].get("make", "")) for k in keys], default="")
    bad_make = gb.add("And", [answered["make"], gb.add("Not", [gb.add(
        "Equal", [labels["make"], rule_make], "hier/makeeq")], "hier/makene")],
        "hier/badmake")

    # `Tuya Plug` ứng với nhiều type nên vế type là TẬP cho phép, không phải đẳng thức.
    n_types = gb.label_encode(
        model_label, keys, "hier/ntypes",
        values=[len(hierarchy[k].get("types", [])) for k in keys], default=0)
    typed = gb.add("Greater", [n_types, gb.const(np.int64([[0]]), "zero")], "hier/typed")
    pair = gb.concat_str([model_label, onnx_lit(ONNX_PAIR_SEP), labels["type"]],
                         "hier/pair")
    pair_keys = [f"{k}{ONNX_PAIR_SEP}{t}"
                 for k in keys for t in hierarchy[k].get("types", [])]
    allowed = gb.add("Cast", [gb.label_encode(pair, pair_keys, "hier/pairhit",
                                              values=[1] * len(pair_keys), default=0)],
                     "hier/allowed", to=TP.BOOL)
    bad_type = gb.add("And", [gb.add("And", [answered["type"], typed], "hier/typechk"),
                              gb.add("Not", [allowed], "hier/nallowed")], "hier/badtype")

    conflict = gb.add("And", [has_rule, gb.add("Or", [bad_make, bad_type], "hier/bad")],
                      "hier/conflict")
    keep = gb.add("Not", [conflict], "hier/keep")
    return {h: gb.add("And", [answered[h], keep], f"final/{h}") for h in labels}

## 6. Tối ưu kích thước

Bốn mảng `class_ids` / `class_nodeids` / `class_treeids` / `class_weights` của
`TreeEnsembleClassifier` là một bảng **thưa** `(node, class) -> weight`; cặp vắng mặt
được runtime coi là 0. RandomForest 25 cây cho lá gần như thuần, nên 92–96% ô đúng bằng
0 — mà skl2onnx vẫn ghi hết. Bỏ chúng **không đổi một chữ số nào** của output, và đây là
phần lớn dung lượng file (~87% mỗi rừng).

`nodes_hitrates` toàn `1.0` và `nodes_missing_value_tracks_true` toàn `0` — đúng giá trị
mặc định trong spec, nên bỏ được; chỉ bỏ sau khi kiểm đúng là vậy chứ không giả định.

Không đụng tới `N_ESTIMATORS`: 03 đã đo 25/100/250 cây cho macro-F1 như nhau, và số cây
quyết định **độ phân giải của confidence**, tức là toàn bộ bảng ngưỡng ở `thresholds.csv`.
Giảm cây nữa thì phải hiệu chỉnh lại ngưỡng, không phải một phép tối ưu file.

In [7]:
def onnx_prune(model):
    """Bỏ trọng số lá bằng 0 và hai thuộc tính đang mang đúng giá trị mặc định."""
    stats = {"weights_kept": 0, "weights_dropped": 0, "attrs_dropped": []}
    for node in model.graph.node:
        if node.op_type not in ("TreeEnsembleClassifier", "TreeEnsembleRegressor"):
            continue
        att = {a.name: a for a in node.attribute}
        weights = np.asarray(att["class_weights"].floats, dtype=np.float32)
        keep = np.flatnonzero(weights != 0)
        stats["weights_kept"] += int(keep.size)
        stats["weights_dropped"] += int(weights.size - keep.size)
        for field in ("class_ids", "class_nodeids", "class_treeids"):
            values = [int(v) for v in np.asarray(att[field].ints)[keep]]
            del att[field].ints[:]
            att[field].ints.extend(values)
        values = [float(v) for v in weights[keep]]
        del att["class_weights"].floats[:]
        att["class_weights"].floats.extend(values)

        hit = att.get("nodes_hitrates")
        miss = att.get("nodes_missing_value_tracks_true")
        drop = ([("nodes_hitrates", hit)] if hit is not None
                and all(v == 1.0 for v in hit.floats) else [])
        drop += ([("nodes_missing_value_tracks_true", miss)] if miss is not None
                 and all(v == 0 for v in miss.ints) else [])
        for name, attr in drop:
            node.attribute.remove(attr)
            stats["attrs_dropped"].append(name)

    model.graph.ClearField("doc_string")
    for node in model.graph.node:
        node.ClearField("doc_string")
    stats["attrs_dropped"] = sorted(set(stats["attrs_dropped"]))
    return stats

## 7. Lắp ráp

Encoder và ba rừng cây được skl2onnx chuyển riêng, gắn tiền tố cho khỏi đụng tên, rồi
nối lại bằng tên tensor. Không dùng `compose.merge_models` vì ở đây cần nối vào những
tensor do chính front sinh ra, không phải ghép hai model đầu-đuôi.

`onnx_set_locale` chạy sau cùng, trên ModelProto đã lắp xong. skl2onnx dựng mỗi cột
TF-IDF bằng một `StringNormalizer` LOWER **không có thuộc tính `locale`**, và onnxruntime
khi thiếu thuộc tính đó sẽ tự lấy `en_US.UTF-8`. Trên máy build thì không sao; trên
router OpenWrt (musl, chỉ có locale `C`) thì `InferenceSession` ném lỗi ngay lúc nạp
model, trước khi chạy được dòng nào. Ghi thẳng `C` vào cả 4 node là cách sửa duy nhất
nằm trong tầm kiểm soát của file model — biến môi trường không đổi được thuộc tính node.

Đặt `C` không đổi kết quả cho bộ feature này: `dns_tokens`, `mdns_tokens`,
`tls_sni_tokens` đã được bước 02 lowercase và chuẩn hoá thành ASCII, `dhcp_vci` trong
toàn bộ tập train cũng thuần ASCII, mà hạ chữ hoa ASCII thì `C` và `en_US.UTF-8` cho
cùng kết quả. Đó là lập luận; mục 9 mới là bằng chứng.

In [8]:
def onnx_relink(graph, rename):
    """Sao node + initializer của một subgraph, đổi tên input theo `rename`."""
    nodes = []
    for node in graph.node:
        copy = onnx.NodeProto()
        copy.CopyFrom(node)
        for i, name in enumerate(copy.input):
            if name in rename:
                copy.input[i] = rename[name]
        nodes.append(copy)
    return nodes, list(graph.initializer)


def onnx_sub_models(bundle):
    """Chuyển encoder và từng rừng cây, gắn tiền tố riêng."""
    enc = bundle["encoder"]
    initial = ([(c, FloatTensorType([None, 1])) for c in enc["num_cols"]]
               + [(c, StringTensorType([None, 1]))
                  for c in enc["cat_cols"] + enc["text_cols"]])
    ct = convert_sklearn(enc["ct"], "sdc_encoder", initial_types=initial,
                         target_opset=ONNX_OPSET)
    # Input giữ nguyên tên cột để front nối vào theo tên; mọi thứ khác gắn tiền tố.
    ct = compose.add_prefix(ct, "enc/", rename_inputs=False)

    n_feat = len(bundle["feature_names"])
    forests = {}
    for head in LABEL_COLS:
        clf = bundle["models"][head]
        m = convert_sklearn(clf, f"sdc_{head}",
                            initial_types=[("X", FloatTensorType([None, n_feat]))],
                            target_opset=ONNX_OPSET,
                            options={id(clf): {"zipmap": False}})
        forests[head] = compose.add_prefix(m, f"rf_{head}/")
    return ct, forests


def onnx_set_locale(model, locale=ONNX_LOCALE):
    """Ghi `locale` cho mọi `StringNormalizer`. Trả tên các node đã sửa.

    Thiếu thuộc tính này thì onnxruntime dựng locale mặc định `en_US.UTF-8` lúc khởi tạo
    session và chết trên hệ musl — xem chú thích ở `ONNX_LOCALE`. Ghi đè chứ không chỉ
    thêm khi vắng: một bản skl2onnx khác có thể đã đặt sẵn tên locale không tồn tại trên
    router, mà file model phải chạy được ở đúng một chỗ.
    """
    patched = []
    for node in model.graph.node:
        if node.op_type != "StringNormalizer":
            continue
        for existing in [a for a in node.attribute if a.name == "locale"]:
            node.attribute.remove(existing)
        node.attribute.append(helper.make_attribute("locale", locale))
        patched.append(node.name)
    return patched


def onnx_assert_locale(model, locale=ONNX_LOCALE):
    """Gác: không node chuỗi nào được rời file mà thiếu locale, hoặc mang locale khác.

    Đây là lỗi chỉ lộ ra trên router chứ không lộ trên máy build, nên phải bắt tại chỗ
    xuất — bắt muộn hơn nghĩa là bắt bằng một lần flash firmware.
    """
    for node in model.graph.node:
        if node.op_type != "StringNormalizer":
            continue
        got = {a.name: a for a in node.attribute}.get("locale")
        assert got is not None, f"{node.name}: thiếu thuộc tính locale"
        assert got.s.decode() == locale, (
            f"{node.name}: locale {got.s.decode()!r}, cần {locale!r}")


def onnx_build_model(bundle, prune=True):
    """Dựng ModelProto hoàn chỉnh. Trả (model, thứ tự cột, thống kê tối ưu)."""
    enc = bundle["encoder"]
    cols = enc["num_cols"] + enc["cat_cols"] + enc["text_cols"]
    ct, forests = onnx_sub_models(bundle)

    gb = OnnxGraphBuilder()
    columns = onnx_build_front(gb, cols, enc["num_cols"])

    nodes, inits = onnx_relink(ct.graph, {i.name: columns[i.name] for i in ct.graph.input})
    gb.nodes += nodes
    gb.inits += inits
    features = ct.graph.output[0].name

    # n_sources = số nguồn có mặt trong cửa sổ, y như `Predictor.predict_row`.
    n_sources = None
    for flag in SOURCE_FLAGS:
        n_sources = (columns[flag] if n_sources is None
                     else gb.add("Add", [n_sources, columns[flag]], "nsrc"))

    labels, answered = {}, {}
    for head in LABEL_COLS:
        forest = forests[head]
        nodes, inits = onnx_relink(forest.graph, {forest.graph.input[0].name: features})
        gb.nodes += nodes
        gb.inits += inits
        raw_label, probs = (o.name for o in forest.graph.output)
        l2_label = gb.add("Reshape", [raw_label, gb.const(np.int64([-1, 1]), "shape")],
                          f"l2label/{head}")
        l1_label = onnx_build_l1(gb, columns, bundle["l1_tables"],
                                 bundle.get("l1_ambiguous", {}), head)
        labels[head], answered[head] = onnx_build_head(
            gb, head, probs, l2_label, l1_label, bundle["models"][head].classes_,
            n_sources, bundle["thresholds_by_source"], bundle["min_sources"],
            bundle["l1_floor"])

    answered = onnx_build_hierarchy(gb, bundle.get("hierarchy", {}), labels, answered)
    abstain = gb.str_const(ONNX_ABSTAIN)
    final = [gb.add("Where", [answered[h], labels[h], abstain], f"out/{h}")
             for h in LABEL_COLS]
    gb.nodes.append(helper.make_node("Concat", final, [ONNX_OUTPUT],
                                     name=gb.name("n_Concat"), axis=1))

    graph = helper.make_graph(
        gb.nodes, "sdc_multihead",
        [helper.make_tensor_value_info(ONNX_INPUT, TP.STRING, [None, len(cols)])],
        [helper.make_tensor_value_info(ONNX_OUTPUT, TP.STRING, [None, len(LABEL_COLS)])],
        gb.inits)

    # skl2onnx dựng TF-IDF bằng `com.microsoft.Tokenizer`, nên opset import phải gom cả
    # domain contrib của onnxruntime — đây cũng là lý do file chỉ chạy trên onnxruntime.
    opsets = {"": ONNX_OPSET, "ai.onnx.ml": ONNX_ML_OPSET}
    for sub in (ct, *forests.values()):
        for imp in sub.opset_import:
            opsets[imp.domain] = max(opsets.get(imp.domain, 0), imp.version)
    model = helper.make_model(
        graph, opset_imports=[helper.make_opsetid(d, v) for d, v in opsets.items()])
    model.ir_version = 10
    model.doc_string = ""

    stats = onnx_prune(model) if prune else {}
    stats["locale_nodes"] = onnx_set_locale(model)
    onnx_assert_locale(model)
    onnx.checker.check_model(model)
    return model, cols, stats


def onnx_contract(bundle, cols, run_id, stats, payload):
    enc = bundle["encoder"]
    return {
        "format": "sdc-onnx-v3",
        "contract_version": ONNX_CONTRACT_VERSION,
        "run_id": run_id,
        "file": ONNX_FILE,
        "bytes": len(payload),
        "runtime": {"required": "onnxruntime",
                    "reason": "TF-IDF dùng com.microsoft.Tokenizer",
                    "opset": ONNX_OPSET, "ai.onnx.ml": ONNX_ML_OPSET,
                    "string_normalizer_locale": ONNX_LOCALE,
                    "locale_note": ("Ghi sẵn trong node nên chạy được trên musl/OpenWrt; "
                                    "không phụ thuộc LANG/LC_ALL của hệ điều hành."),
                    },
        "io": {
            "input": {
                "name": ONNX_INPUT, "type": "string", "shape": ["N", len(cols)],
                "columns": cols,
                "note": ("Cột numeric gửi dưới dạng chuỗi số ('1', '0'); nguồn vắng mặt "
                         "-> numeric '0', cat/text '<missing>'. Không impute."),
            },
            "output": {
                "name": ONNX_OUTPUT, "type": "string", "shape": ["N", len(LABEL_COLS)],
                "heads": list(LABEL_COLS), "abstain_label": ONNX_ABSTAIN,
            },
        },
        "labels": {h: [str(c) for c in bundle["models"][h].classes_] for h in LABEL_COLS},
        "policy_in_graph": {
            "tiers": ["l1_mined", "l2_threshold", "hierarchy"],
            "thresholds_by_source": bundle["thresholds_by_source"],
            "min_sources": bundle["min_sources"],
            "l1_floor": bundle["l1_floor"],
            "comparison_epsilon": ONNX_EPS,
            "note": "Đổi ngưỡng phải xuất lại file; sửa contract.json không có tác dụng.",
        },
        "policy_outside_graph": {
            # `as_posix` chứ không phải `str`: contract.json đi kèm model sang máy khác.
            "enrolled": ENROLLED_PATH.relative_to(ROOT).as_posix(),
            "semantic_type_catalog": DEFAULT_CATALOG.relative_to(ROOT).as_posix(),
            "note": ("Graph tái lập Predictor(run_dir, enrolled={}, "
                     "semantic_catalog=None). Hai tầng này chạy ngoài nếu dùng."),
        },
        "columns": {"num": enc["num_cols"], "cat": enc["cat_cols"],
                    "text": enc["text_cols"]},
        "size_optimization": stats,
    }


def onnx_export(run_dir, prune=True):
    """Ghi `sdc_multihead.onnx` + `contract.json` vào thư mục run. Trả đường dẫn file."""
    run_dir = Path(run_dir)
    bundle = joblib.load(run_dir / "model.joblib")
    model, cols, stats = onnx_build_model(bundle, prune=prune)
    payload = model.SerializeToString()
    path = run_dir / ONNX_FILE
    path.write_bytes(payload)
    (run_dir / "contract.json").write_text(
        json.dumps(onnx_contract(bundle, cols, run_dir.name, stats, payload),
                   indent=2, ensure_ascii=False), encoding="utf-8")
    return path

## 8. Xuất candidate

`ONNX_RUN = None` lấy run đang ghim trong `Models/current_model.json`, hoặc candidate duy
nhất nếu chưa kích hoạt cái nào. Bảng dưới đo luôn phần tiết kiệm được: cùng graph, chỉ
khác `prune`.

In [9]:
ONNX_RUN = None        # ví dụ: "20260913_230058_verified_tiered"

onnx_run_dir = select_run_dir(ONNX_RUN or globals().get("run_dir"))
onnx_path = onnx_export(onnx_run_dir)
onnx_contract_doc = json.loads((onnx_run_dir / "contract.json").read_text(encoding="utf-8"))

_raw, _, _ = onnx_build_model(joblib.load(onnx_run_dir / "model.joblib"), prune=False)
_raw_bytes = len(_raw.SerializeToString())
_stats = onnx_contract_doc["size_optimization"]

print(f"run        {onnx_run_dir.name}")
print(f"model.joblib   {(onnx_run_dir / 'model.joblib').stat().st_size:>9,} bytes")
print(f"onnx chưa cắt  {_raw_bytes:>9,} bytes")
print(f"onnx đã cắt    {onnx_path.stat().st_size:>9,} bytes  "
      f"(-{1 - onnx_path.stat().st_size / _raw_bytes:.1%})")
print(f"trọng số lá giữ {_stats['weights_kept']:,} / "
      f"{_stats['weights_kept'] + _stats['weights_dropped']:,}")
display(pd.DataFrame([onnx_contract_doc["io"]["input"], onnx_contract_doc["io"]["output"]],
                     index=["input", "output"])[["name", "type", "shape"]])

run        20260914_154821_verified_tiered
model.joblib     429,796 bytes
onnx chưa cắt  1,366,446 bytes
onnx đã cắt      347,747 bytes  (-74.6%)
trọng số lá giữ 4,816 / 90,634


,name,type,shape
input,input,string,"[N, 40]"
output,output,string,"[N, 3]"


## 9. Parity với `Predictor`

Đây là cell quyết định file dùng được hay không. Mốc so sánh là
`Predictor(run_dir, enrolled={}, semantic_catalog=None)` — tắt đúng hai tầng cố ý để
ngoài graph. So trên **từng session** của tập đã xác minh, cả ba head.

Lệch ở đây không phải sai số cho phép: nó nghĩa là graph và runtime python đang chạy hai
policy khác nhau, và `04`/`06` đã đo bản python.

In [10]:
def onnx_to_input(frame, cols):
    """40 cột -> một tensor chuỗi [N,40], đúng thứ tự trong contract."""
    out = frame[cols].copy()
    for col in cols:
        values = out[col]
        out[col] = (values.astype(np.float32).astype(str)
                    if pd.api.types.is_numeric_dtype(values) else values.astype(str))
    return out.to_numpy(dtype=object)


def onnx_reference(run_dir, frame, heads):
    """Nhãn của `Predictor` cho từng dòng: nhãn đã trả lời, hoặc `unknown`.

    `n_jobs = 1` cho vòng này: `make_model()` đặt `n_jobs=-1` để train nhanh, nhưng ở đây
    mỗi lần gọi chỉ có MỘT dòng, nên chi phí dựng thread pool lớn hơn cả 25 cây. Không
    đụng tới model trên đĩa, chỉ đối tượng vừa nạp.
    """
    predictor = Predictor(run_dir, enrolled={}, semantic_catalog=None)
    for clf in predictor.models.values():
        clf.n_jobs = 1
    rows = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # một dòng/lần -> sklearn cảnh báo mỗi lần gọi
        for record in frame.to_dict("records"):
            out = predictor.predict_row(record)
            rows.append([out[h]["top1"] if out[h]["status"] == "answer" else ONNX_ABSTAIN
                         for h in heads])
    return np.array(rows, dtype=object)


def onnx_verify(run_dir, contract, limit=None, seed=0):
    cols = contract["io"]["input"]["columns"]
    heads = contract["io"]["output"]["heads"]
    frame = normalize_heads(pd.read_parquet(SESSIONS_PATH))
    if limit and limit < len(frame):
        frame = frame.sample(limit, random_state=seed).reset_index(drop=True)

    session = ort.InferenceSession(str(run_dir / contract["file"]),
                                   providers=["CPUExecutionProvider"])
    got = session.run(None, {contract["io"]["input"]["name"]:
                             onnx_to_input(frame, cols)})[0]
    want = onnx_reference(run_dir, frame, heads)

    rows, mismatch = [], {}
    for i, head in enumerate(heads):
        same = got[:, i] == want[:, i]
        rows.append({"head": head, "n": len(frame), "khớp": float(same.mean()),
                     "lệch": int((~same).sum()),
                     "trả lời": int((got[:, i] != ONNX_ABSTAIN).sum())})
        if not same.all():
            bad = frame.loc[~same, ["canonical_device", head]].copy()
            bad["onnx"] = got[~same, i]
            bad["predictor"] = want[~same, i]
            mismatch[head] = bad
    return pd.DataFrame(rows).set_index("head"), mismatch


ONNX_VERIFY_LIMIT = None     # None = toàn bộ session; đặt số để chạy nhanh khi sửa code

onnx_parity, onnx_mismatch = onnx_verify(onnx_run_dir, onnx_contract_doc,
                                         limit=ONNX_VERIFY_LIMIT)
display(onnx_parity)
for head, bad in onnx_mismatch.items():
    print(f"\n{head}: {len(bad)} session lệch")
    display(bad.head(15))
assert not onnx_mismatch, "ONNX và Predictor không cùng policy — xem bảng lệch ở trên"
print("Parity OK — graph và Predictor cho cùng nhãn trên mọi session")

,n,khớp,lệch,trả lời
head,,,,
make,8139,1.0,0,6088
type,8139,1.0,0,6093
model,8139,1.0,0,6106


Parity OK — graph và Predictor cho cùng nhãn trên mọi session
